In [1]:
from datasets import load_dataset
from huggingface_hub import login
from tqdm import tqdm

In [2]:
login()

In [3]:
dataset = load_dataset(
    "harsha-desaraju/telugu-sanskrit-english-text"
)["train"]
dataset

README.md:   0%|          | 0.00/377 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

data/train-00000-of-00033.parquet:   0%|          | 0.00/198M [00:00<?, ?B/s]

data/train-00001-of-00033.parquet:   0%|          | 0.00/198M [00:00<?, ?B/s]

...

data/train-00031-of-00033.parquet:   0%|          | 0.00/296M [00:00<?, ?B/s]

data/train-00032-of-00033.parquet:   0%|          | 0.00/297M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4111832 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/32 [00:00<?, ?it/s]

Dataset({
    features: ['text', 'lang', 'num_words'],
    num_rows: 4111832
})

In [4]:
import re
import regex
import unicodedata
from collections import Counter

def split_graphemes_english(sample):
    text = sample['text']

    # Decompose compound characters and remove non-ascii
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    
    graphemes = regex.findall(r'\X', text)
    sample['graphemes_count'] = dict(Counter(graphemes))
    return sample


def split_graphemes_indic(sample):
    text = sample['text']

    # Combine compound characters
    text = unicodedata.normalize("NFKC", text)

    # Keep only telugu unicode and ascii only
    text = re.sub(r"[^\u0C00-\u0C7F\x00-\x7F]+", "", text)

    graphemes = regex.findall(r'\X', text)
    sample['graphemes_count'] = dict(Counter(graphemes))
    return sample
    

In [5]:
san_ds = dataset.filter(lambda x: x['lang']=='sanskrit')
eng_ds = dataset.filter(lambda x: x['lang']=='english')
tel_ds = dataset.filter(lambda x: x['lang']=='telugu')

Filter:   0%|          | 0/4111832 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4111832 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4111832 [00:00<?, ? examples/s]

In [6]:
eng_ds = eng_ds.map(split_graphemes_english, num_proc=4)
san_ds = san_ds.map(split_graphemes_indic, num_proc=4)
tel_ds = tel_ds.map(split_graphemes_indic, num_proc=4)

tel_ds

Map (num_proc=4):   0%|          | 0/415045 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/740210 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2956577 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'lang', 'num_words', 'graphemes_count'],
    num_rows: 2956577
})

In [7]:
def get_dist(ds):
    dist = Counter(ds[0]['graphemes_count'])
    for sample in tqdm(ds):
        dist += Counter(sample['graphemes_count'])
    return dist

In [8]:
import gc
import json

In [9]:
%%time

print(len(eng_ds))
eng_dist = get_dist(eng_ds)
print(f"No.of unique graphemes in English: {len(eng_dist)}")

eng_dist = dict(eng_dist)
with open("token_dist/english_grapheme_dist.json", 'w') as f:
    json.dump(eng_dist, f, indent=4)

del eng_dist
del eng_ds
gc.collect()

415045


100%|██████████| 415045/415045 [00:46<00:00, 8985.81it/s] 


No.of unique graphemes in English: 75
CPU times: user 42.8 s, sys: 1.16 s, total: 44 s
Wall time: 46.4 s


50

In [10]:
%%time

print(len(san_ds))
san_dist = get_dist(san_ds)
print(f"No.of unique graphemes in Sanskrit: {len(san_dist)}")

san_dist = dict(san_dist)
with open("token_dist/sanskrit_grapheme_dist.json", 'w') as f:
    json.dump(san_dist, f, indent=4)

del san_dist
del san_ds
gc.collect()

740210


100%|██████████| 740210/740210 [13:14<00:00, 932.23it/s] 


No.of unique graphemes in Sanskrit: 43753
CPU times: user 13min 7s, sys: 8.02 s, total: 13min 15s
Wall time: 13min 14s


33

In [12]:
NUM_PROC = 4

batch_size = len(tel_ds) // NUM_PROC

datasets = []
for i in range(0, len(tel_ds), batch_size):
    datasets.append(tel_ds.select(range(i, min(i+batch_size, len(tel_ds)))))

print(len(datasets))

5


In [ ]:
# %% time

for i, small_tel_ds in enumerate(datasets, 1):
    tel_dist = get_dist(small_tel_ds)
    print(f"No.of unique graphemes in Telugu sample {i}: {len(tel_dist)}")
    
    tel_dist = dict(tel_dist)
    with open(f"token_dist/telugu_grapheme_dist_{i}.json", 'w') as f:
        json.dump(tel_dist, f, indent=4)
    
    del tel_dist
    # del tel_ds
    gc.collect()

 44%|████▍     | 326270/739144 [07:45<13:31, 508.96it/s]